<a href="https://colab.research.google.com/github/phenomenaldatalab/Foundations-of-AI-for-Business/blob/main/Exercise7_Reasoning_about_the_Physical_World.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Search and TMS

In [ ]:
import itertools
import copy

class SudokuSolverTMS:
    def __init__(self, puzzle):
        # The initial puzzle state
        self.puzzle = puzzle
        # Initialize the domains for each cell
        # If the cell is empty (0), the domain is {1, 2, 3, 4, 5, 6, 7, 8, 9}
        # If the cell is filled, the domain is just the filled value
        self.domains = [[{1, 2, 3, 4, 5, 6, 7, 8, 9} if puzzle[i][j] == 0 else {puzzle[i][j]} for j in range(9)] for i in range(9)]
        # Initialize the TMS (Truth Maintenance System) to keep track of changes and dependencies
        self.tms = [[[] for _ in range(9)] for _ in range(9)]

    def print_puzzle(self):
        # Print the current state of the puzzle
        for row in self.puzzle:
            print(" ".join(str(num) if num != 0 else '.' for num in row))
        print("\n")

    def propagate_constraints(self):
        # Propagate constraints to reduce domains
        # This method narrows down the possible values for each cell by removing values that are no longer valid
        changed = True
        while changed:
            changed = False
            for i in range(9):
                for j in range(9):
                    # If a cell has only one possible value, propagate that value to its peers
                    if len(self.domains[i][j]) == 1:
                        value = next(iter(self.domains[i][j]))
                        # Remove this value from all peers of the current cell
                        for peer in self.get_peers(i, j):
                            if value in self.domains[peer[0]][peer[1]]:
                                # print(f"Removing {value} from ({peer[0]},{peer[1]}) due to ({i},{j}) being {value}")
                                self.domains[peer[0]][peer[1]].remove(value)
                                # Record this change in the TMS for potential backtracking
                                self.tms[peer[0]][peer[1]].append(f"Removed {value} due to ({i},{j}) being {value}")
                                changed = True


    def get_peers(self, row, col):
        # Get all peers of the given cell (row, col)
        peers = set()
        # Row and column peers
        for k in range(9):
            if k != col:
                peers.add((row, k))
            if k != row:
                peers.add((k, col))
        # Box peers
        box_row_start, box_col_start = 3 * (row // 3), 3 * (col // 3)
        for i in range(box_row_start, box_row_start + 3):
            for j in range(box_col_start, box_col_start + 3):
                if (i, j) != (row, col):
                    peers.add((i, j))
        return peers

    def is_solved(self):
        # Check if the puzzle is solved
        # The puzzle is solved if every cell has exactly one value in its domain
        return all(len(self.domains[i][j]) == 1 for i in range(9) for j in range(9))

    def guess(self):
        # Find a cell with more than one possibility and make a guess
        for i in range(9):
            for j in range(9):
                if len(self.domains[i][j]) > 1:
                    # Try each value in the domain of the cell
                    for guess in self.domains[i][j]:
                        print(f"Guessing {guess} for cell ({i},{j})")
                        # Save the current state of the domains to allow backtracking
                        saved_domains = copy.deepcopy(self.domains)
                        # Set the domain of the current cell to the guessed value
                        self.domains[i][j] = {guess}
                        # Propagate the constraints after making the guess
                        self.propagate_constraints()
                        # Check if the puzzle is solved
                        if self.is_solved():
                            return True
                        else:
                            # If a contradiction is found, backtrack
                            print(f"Backtracking on guess {guess} for cell ({i},{j})")
                            self.domains = saved_domains
                    # If no guess leads to a solution, return False
                    return False
        # If all cells have a single value, return True
        return False

    def solve(self):
        # Print the initial state of the puzzle
        print("Initial Puzzle:")
        self.print_puzzle()

        # Apply constraint propagation and guessing until the puzzle is solved
        while not self.is_solved():
            # First, try to solve using constraint propagation
            self.propagate_constraints()
            # If the puzzle is not yet solved, make a guess
            if not self.is_solved():
                if not self.guess():
                    print("No solution found.")
                    return

        # Print the solved puzzle
        print("Solved Puzzle:")
        self.puzzle = [[next(iter(self.domains[i][j])) for j in range(9)] for i in range(9)]
        self.print_puzzle()

# Example Sudoku puzzle (0 represents an empty cell)
puzzle = [
    [0, 0, 7, 6, 0, 0, 0, 3, 4],
    [2, 8, 9, 0, 0, 4, 0, 0, 0],
    [3, 4, 6, 2, 0, 5, 0, 9, 0],
    [6, 0, 2, 0, 0, 0, 0, 1, 0],
    [0, 3, 8, 0, 0, 6, 0, 4, 7],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 9, 0, 0, 0, 0, 0, 7, 8],
    [7, 0, 3, 4, 0, 0, 5, 6, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0]
]

solver = SudokuSolverTMS(puzzle)
solver.solve()


Initial Puzzle:
. . 7 6 . . . 3 4
2 8 9 . . 4 . . .
3 4 6 2 . 5 . 9 .
6 . 2 . . . . 1 .
. 3 8 . . 6 . 4 7
. . . . . . . . .
. 9 . . . . . 7 8
7 . 3 4 . . 5 6 .
. . . . . . . . .


Guessing 1 for cell (0,0)
Backtracking on guess 1 for cell (0,0)
Guessing 5 for cell (0,0)
Solved Puzzle:
5 1 7 6 9 8 2 3 4
2 8 9 1 3 4 7 5 6
3 4 6 2 7 5 8 9 1
6 7 2 8 4 9 3 1 5
1 3 8 5 2 6 9 4 7
9 5 4 7 1 3 6 8 2
4 9 5 3 6 2 1 7 8
7 2 3 4 8 1 5 6 9
8 6 1 9 5 7 4 2 3




# TMS

In [ ]:
import itertools

class SudokuCSP:
    def __init__(self, grid):
        self.grid = grid
        self.variables = [(r, c) for r in range(9) for c in range(9)]
        self.domains = {var: set(range(1, 10)) if self.grid[var[0]][var[1]] == 0 else {self.grid[var[0]][var[1]]} for var in self.variables}
        self.constraints = self._generate_constraints()

    def _generate_constraints(self):
        constraints = []
        for i in range(9):
            # Row and column constraints
            constraints.append([(i, j) for j in range(9)])
            constraints.append([(j, i) for j in range(9)])
        # Block constraints
        for br, bc in itertools.product(range(0, 9, 3), range(0, 9, 3)):
            constraints.append([(br + r, bc + c) for r in range(3) for c in range(3)])
        return constraints

    def is_consistent(self, assignment):
        for group in self.constraints:
            seen = set()
            for var in group:
                if var in assignment:
                    if assignment[var] in seen:
                        return False
                    seen.add(assignment[var])
        return True

    def infer(self):
        """Propagate constraints to narrow domains."""
        changes = True
        while changes:
            changes = False
            for group in self.constraints:
                # Gather already assigned values
                assigned = {self.grid[var[0]][var[1]] for var in group if len(self.domains[var]) == 1}
                for var in group:
                    if len(self.domains[var]) > 1:
                        before = len(self.domains[var])
                        self.domains[var] -= assigned
                        if len(self.domains[var]) != before:
                            changes = True

    def backtracking_search(self, assignment={}):
        if len(assignment) == len(self.variables):
            return assignment

        var = min((v for v in self.variables if v not in assignment), key=lambda v: len(self.domains[v]))
        for value in self.domains[var]:
            local_assignment = assignment.copy()
            local_assignment[var] = value
            if self.is_consistent(local_assignment):
                result = self.backtracking_search(local_assignment)
                if result:
                    return result

        return None

    def solve(self):
        self.infer()
        assignment = self.backtracking_search()
        if assignment:
            solution_grid = [[0] * 9 for _ in range(9)]
            for (r, c), value in assignment.items():
                solution_grid[r][c] = value
            self.print_solution(solution_grid)
        else:
            print("No solution exists.")

    @staticmethod
    def print_solution(grid):
        for row in grid:
            print(" ".join(map(str, row)))

# Example Usage
grid = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

solver = SudokuCSP(grid)
solver.solve()


5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


In [ ]:
class TruthMaintenanceSystem:
    def __init__(self):
        self.variables = {}  # Each variable and its domain
        self.dependencies = {}  # Dependency relationships between variables

    def add_variable(self, var, domain):
        """Add a variable with its domain to the TMS."""
        self.variables[var] = domain
        self.dependencies[var] = []

    def add_dependency(self, var, dependency, condition):
        """Add a dependency to a variable with a condition for domain reduction."""
        self.dependencies[var].append((dependency, condition))

    def resolve(self):
        """Propagate domain constraints by resolving dependencies."""
        updated = True
        while updated:
            updated = False
            for var, deps in self.dependencies.items():
                if len(self.variables[var]) == 1:  # If the domain is reduced to a single value
                    value = next(iter(self.variables[var]))  # Get the single value
                    for dep, condition in deps:  # Check dependencies
                        if condition(value):
                            before = len(self.variables[dep])
                            self.variables[dep] = self.variables[dep] - {value}  # Remove value from dependent variable's domain
                            if len(self.variables[dep]) != before:
                                print(f"Updated domain of {dep}: {self.variables[dep]} due to {var} = {value}")
                                updated = True


class SudokuWithTMS:
    def __init__(self, grid):
        self.grid = grid
        self.variables = [(r, c) for r in range(9) for c in range(9)]
        self.domains = {var: set(range(1, 10)) if self.grid[var[0]][var[1]] == 0 else {self.grid[var[0]][var[1]]} for var in self.variables}
        self.constraints = self._generate_constraints()
        self.tms = TruthMaintenanceSystem()  # Initialize TMS

    def _generate_constraints(self):
        """Generate all constraints (rows, columns, 3x3 blocks)."""
        constraints = []
        for i in range(9):
            # Row and column constraints
            constraints.append([(i, j) for j in range(9)])
            constraints.append([(j, i) for j in range(9)])
        # Block constraints
        for br, bc in itertools.product(range(0, 9, 3), range(0, 9, 3)):
            constraints.append([(br + r, bc + c) for r in range(3) for c in range(3)])
        return constraints

    def setup_tms(self):
        """Set up the TMS with variables, domains, and dependencies."""
        for var in self.variables:
            self.tms.add_variable(var, self.domains[var])

        for group in self.constraints:
            for var in group:
                for neighbor in group:
                    if var != neighbor:
                        # Dependency: a variable depends on its neighbors not having the same value
                        self.tms.add_dependency(var, neighbor, lambda x: x in self.domains[neighbor])

    def infer(self):
        """Propagate constraints to narrow down the domains."""
        print("Starting inference...")
        changes = True
        while changes:
            changes = False
            for group in self.constraints:
                # Gather already assigned values
                assigned = {self.grid[var[0]][var[1]] for var in group if len(self.domains[var]) == 1}
                for var in group:
                    if len(self.domains[var]) > 1:
                        before = len(self.domains[var])
                        self.domains[var] -= assigned
                        if len(self.domains[var]) != before:
                            print(f"Reduced domain of {var}: {self.domains[var]}")
                            changes = True

    def solve_with_tms(self):
        """Solve the puzzle using TMS-based propagation and backtracking."""
        self.infer()  # First pass at narrowing domains
        self.setup_tms()  # Set up TMS with dependencies
        self.tms.resolve()  # Resolve all dependencies to propagate constraints further
        return self.backtracking_search()

    def is_consistent(self, assignment):
        """Check if an assignment satisfies all constraints."""
        for group in self.constraints:
            seen = set()
            for var in group:
                if var in assignment:
                    if assignment[var] in seen:
                        return False
                    seen.add(assignment[var])
        return True

    def backtracking_search(self, assignment={}):
        """Recursive backtracking search for a solution."""
        if len(assignment) == len(self.variables):  # All variables assigned
            return assignment

        # Select the variable with the smallest domain (minimum remaining values heuristic)
        var = min((v for v in self.variables if v not in assignment), key=lambda v: len(self.domains[v]))
        for value in self.domains[var]:
            local_assignment = assignment.copy()
            local_assignment[var] = value
            if self.is_consistent(local_assignment):
                result = self.backtracking_search(local_assignment)
                if result:
                    return result

        return None

    def print_solution(self, solution):
        """Print the Sudoku grid in a nice format."""
        print("\nSolved Puzzle:")
        grid = [[0] * 9 for _ in range(9)]
        for (r, c), value in solution.items():
            grid[r][c] = value
        for row in grid:
            print(" ".join(map(str, row)))


# Example Usage
grid = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

solver_tms = SudokuWithTMS(grid)
solution_tms = solver_tms.solve_with_tms()
solver_tms.print_solution(solution_tms)


Starting inference...
Reduced domain of (0, 2): {1, 2, 4, 6, 8, 9}
Reduced domain of (0, 3): {1, 2, 4, 6, 8, 9}
Reduced domain of (0, 5): {1, 2, 4, 6, 8, 9}
Reduced domain of (0, 6): {1, 2, 4, 6, 8, 9}
Reduced domain of (0, 7): {1, 2, 4, 6, 8, 9}
Reduced domain of (0, 8): {1, 2, 4, 6, 8, 9}
Reduced domain of (2, 0): {1, 2, 3, 9}
Reduced domain of (6, 0): {1, 2, 3, 9}
Reduced domain of (7, 0): {1, 2, 3, 9}
Reduced domain of (8, 0): {1, 2, 3, 9}
Reduced domain of (1, 1): {2, 3, 4, 7, 8}
Reduced domain of (1, 2): {2, 3, 4, 7, 8}
Reduced domain of (1, 6): {2, 3, 4, 7, 8}
Reduced domain of (1, 7): {2, 3, 4, 7, 8}
Reduced domain of (1, 8): {2, 3, 4, 7, 8}
Reduced domain of (1, 1): {2, 4, 7, 8}
Reduced domain of (3, 1): {1, 2, 4, 5, 7, 8}
Reduced domain of (4, 1): {1, 2, 4, 5, 7, 8}
Reduced domain of (5, 1): {1, 2, 4, 5, 7, 8}
Reduced domain of (7, 1): {1, 2, 4, 5, 7, 8}
Reduced domain of (8, 1): {1, 2, 4, 5, 7, 8}
Reduced domain of (2, 0): {1, 2, 3}
Reduced domain of (2, 3): {1, 2, 3, 4, 5, 